Cell 1: Setup och Bibliotek

Förklaring: Vi laddar verktygen för att prata med AI (requests), hantera data (pandas), bygga databasen (chromadb) och träna student-modellen (sklearn).

In [ ]:
# Installation (körs en gång)
# %pip install chromadb pandas scikit-learn joblib requests

import json, random, requests, sqlite3, joblib
import pandas as pd
from datetime import datetime
import chromadb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

Cell 2: RAG-databasen (Kunskap & Ekonomi)
Förklaring: Här bygger vi "biblioteket". Vi mappar IP-adresser till kunder och kopplar dem till finansiella straffavgifter (SLA).

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="corporate_policy")

# Vi matar in teknisk policy, kunddata och ekonomiska SLA-regler
knowledge = [
    {"id": "p1", "text": "DDoS på 10.0.0.1 (>500 pkt/s): Isolera. Action: ISO-99. Kund: Global Bank (Guld)."},
    {"id": "p2", "text": "IP 192.168.1.100: IoT-enhet för Lilla Caféet (Brons). Extern IP-kontakt förbjuden."},
    {"id": "f1", "text": "SLA Guld kostnad: 5000 SEK/min vid avbrott. SLA Brons: 200 SEK/min."},
    {"id": "a1", "text": "Action ISO-99 innebär total nätverksisolering och tekniker-larm."}
]

collection.add(
    documents=[k["text"] for k in knowledge],
    ids=[k["id"] for k in knowledge]
)

Cell 3: Läraren (Gemma) & Datainsamling
Förklaring: Gemma agerar lärare. Den läser policyn i ChromaDB, tittar på trafiken och fattar beslut. Allt sparas som träningsdata i en .jsonl-fil.


In [ ]:
def ask_agent(data):
    # RAG: Hämta relevant kontext
    res = collection.query(query_texts=[str(data)], n_results=3)
    context = "\n".join(res['documents'][0])
    
    prompt = f"Policy:\n{context}\nData: {json.dumps(data)}\nSvara strikt JSON: {{'decision': 'isolate'/'none', 'customer': 'namn', 'sla': 'Guld'/'Silver'/'Brons', 'action_code': 'kod', 'reason': 'varför'}}"
    
    resp = requests.post("http://localhost:11434/api/generate", 
                         json={"model": "gemma2", "prompt": prompt, "stream": False, "format": "json"})
    return json.loads(resp.json()['response'])

# Simulera 100 rader data
logs = []
for tick in range(1, 101):
    traffic = random.randint(10, 800) if tick % 10 == 0 else random.randint(10, 50)
    # Skapa en datapunkt
    telemetry = {"10.0.0.1": {"traffic": traffic, "notes": "Stable"}}
    decision = ask_agent(telemetry)
    logs.append({**telemetry, **decision, "target": 1 if decision['decision'] == 'isolate' else 0})

df = pd.DataFrame(logs)
df.to_json("audit_log.jsonl", orient="records", lines=True)

Cell 4: Träna Studenten (ML-Modellen) 
Förklaring: Vi destillerar Gemmas kunskap till en snabb modell. Vi använder $F_1$-score som mätvärde för att balansera precision och recall.

In [ ]:
# Feature Engineering
X = pd.DataFrame()
X['traffic'] = df['10.0.0.1'].apply(lambda x: x['traffic'])
y = df['target']

# Träning med Stratify (säkerställer att attacker finns i både träning och test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

grid = GridSearchCV(HistGradientBoostingClassifier(), {'learning_rate': [0.1, 0.01]}, cv=3)
grid.fit(X_train, y_train)

# Spara den färdiga studenten
joblib.dump(grid.best_estimator_, "student_model.pkl")
print("Modellen sparad som student_model.pkl")